In [1]:
# Cell 1: Imports and Config
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import inception_v3, Inception_V3_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

from torch.utils.tensorboard import SummaryWriter

# ================= CONFIG =================
SEED = 42

BATCH_SIZE = 128  # smaller batch due to inception memory
EPOCHS = 10
LR = 3e-4
IMG_SIZE = 299  # Inception default input size

NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

torch.backends.cudnn.benchmark = True

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

2026-02-27 14:26:19.987224: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772202380.316045      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772202380.483976      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772202381.308205      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772202381.308248      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772202381.308251      55 computation_placer.cc:177] computation placer alr

CUDA available: True
GPU count: 2


In [2]:
# Cell 2: Data Collection
NO_PATHS = [
    "/kaggle/input/datasets/navoneel/brain-mri-images-for-brain-tumor-detection/no",
    "/kaggle/input/datasets/ahmedhamada0/brain-tumor-detection/no",
]

YES_PATHS = [
    "/kaggle/input/datasets/navoneel/brain-mri-images-for-brain-tumor-detection/yes",
    "/kaggle/input/datasets/ahmedhamada0/brain-tumor-detection/yes",
]

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".tif")


def collect(paths, label):
    items = []
    for path in paths:
        for root, _, files in os.walk(path):
            for f in files:
                if f.lower().endswith(VALID_EXTENSIONS):
                    items.append((os.path.join(root, f), label))
    return items


data = collect(NO_PATHS, 0) + collect(YES_PATHS, 1)
df = pd.DataFrame(data, columns=["path", "label"])

print("Total Images:", len(df))
print("Class Distribution:")
print(df["label"].value_counts())

Total Images: 3253
Class Distribution:
label
1    1655
0    1598
Name: count, dtype: int64


In [3]:
# Cell 3: Train/Val/Test Split
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 2277
Val: 488
Test: 488


In [4]:
# Cell 4: Dataset Class
class BrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "path"]
        label = self.df.loc[idx, "label"]

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.float32)

In [5]:
# Cell 5: Transforms and DataLoaders
train_tfms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

val_tfms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

train_loader = DataLoader(
    BrainDataset(train_df, train_tfms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=True,
)

val_loader = DataLoader(
    BrainDataset(val_df, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=False,
)

test_loader = DataLoader(
    BrainDataset(test_df, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=False,
)

In [6]:
# Cell 6: Model Setup
model = inception_v3(weights=Inception_V3_Weights.DEFAULT, aux_logits=True)
model.fc = nn.Linear(model.fc.in_features, 1)
model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, 1)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler()

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 231MB/s] 


In [7]:
def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    loop = tqdm(
        loader, total=len(loader), desc="Train" if training else "Val/Test", leave=False
    )

    for images, labels in loop:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(
            DEVICE, non_blocking=True
        )

        with torch.amp.autocast(device_type="cuda", enabled=True):
            # Use model.module if DataParallel
            m = model.module if isinstance(model, nn.DataParallel) else model

            if training and model.training and getattr(m, "aux_logits", False):
                outputs, aux_outputs = m(images)
                loss1 = criterion(outputs.squeeze(1), labels)
                loss2 = criterion(aux_outputs.squeeze(1), labels)
                loss = loss1 + 0.4 * loss2
            else:
                outputs = m(images)
                loss = criterion(outputs.squeeze(1), labels)

        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item()
        probs = torch.sigmoid(outputs).detach()
        all_preds.append(probs.cpu())
        all_labels.append(labels.detach().cpu())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = accuracy_score(all_labels, all_preds > 0.5)
    auc = roc_auc_score(all_labels, all_preds)
    return total_loss / len(loader), acc, auc

In [8]:
# Cell 8: Training Loop
log_dir = f"/kaggle/working/runs/{datetime.now().strftime('%Y%m%d-%H%M%S')}"
writer = SummaryWriter(log_dir)
best_auc = 0.0

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")

    train_loss, train_acc, train_auc = run_epoch(train_loader, True)
    val_loss, val_acc, val_auc = run_epoch(val_loader, False)

    scheduler.step()

    print(
        f"Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f}"
    )
    print(f"Val   | Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}")

    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("Loss/Val", val_loss, epoch)
    writer.add_scalar("AUC/Train", train_auc, epoch)
    writer.add_scalar("AUC/Val", val_auc, epoch)

    if val_auc > best_auc:
        best_auc = val_auc
        state_dict = (
            model.module.state_dict()
            if torch.cuda.device_count() > 1
            else model.state_dict()
        )
        torch.save(state_dict, "/kaggle/working/best_inception.pth")
        print("Best model saved.")

writer.close()


===== Epoch 1/10 =====


Train:   0%|          | 0/17 [00:00<?, ?it/s]

Val/Test:   0%|          | 0/4 [00:00<?, ?it/s]

Train | Loss: 0.3156 | Acc: 0.8980 | AUC: 0.9699
Val   | Loss: 0.0235 | Acc: 0.9918 | AUC: 0.9997
Best model saved.

===== Epoch 2/10 =====


Train:   0%|          | 0/17 [00:00<?, ?it/s]

...

Val/Test:   0%|          | 0/4 [00:00<?, ?it/s]

Train | Loss: 0.0014 | Acc: 1.0000 | AUC: 1.0000
Val   | Loss: 0.0288 | Acc: 0.9939 | AUC: 0.9999

===== Epoch 10/10 =====


Train:   0%|          | 0/17 [00:00<?, ?it/s]

Val/Test:   0%|          | 0/4 [00:00<?, ?it/s]

Train | Loss: 0.0012 | Acc: 1.0000 | AUC: 1.0000
Val   | Loss: 0.0280 | Acc: 0.9939 | AUC: 0.9999


In [9]:
# Cell 9: Load Best Model & Evaluate on Test Set
state_dict = torch.load("/kaggle/working/best_inception.pth")
if torch.cuda.device_count() > 1:
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()
test_loss, test_acc, test_auc = run_epoch(test_loader, training=False)

print("\n===== FINAL UNSEEN TEST RESULTS =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

Val/Test:   0%|          | 0/4 [00:00<?, ?it/s]


===== FINAL UNSEEN TEST RESULTS =====
Test Loss: 0.0073
Test Accuracy: 0.9939
Test AUC: 1.0000


In [10]:
# Cell 10: Save Final Model
final_state = (
    model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict()
)
torch.save(final_state, "/kaggle/working/final_inception_brain_tumor.pth")
print("Final model saved.")

Final model saved.
